
# Multi-Class Workout Recommendation System
## Scraped-Style Dataset + Preprocessing + Explainable ML

This notebook implements:

- realistic scraped-data preprocessing
- noisy data cleaning
- multiclass workout recommendation
- machine learning comparison
- XGBoost training
- feature importance analysis
- explainable AI workflow

Target classes:
- cardio
- strength
- yoga
- hiit
- bodyweight
- crossfit


In [ ]:

# INSTALL LIBRARIES

!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn joblib


In [ ]:

# IMPORTS

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import shap
import joblib

import warnings
warnings.filterwarnings('ignore')


In [ ]:

# LOAD DATASET

DATA_PATH = 'scraped_style_noisy_workout_dataset.csv'

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()


In [ ]:

# DATASET OVERVIEW

print(df.info())

print(df.isnull().sum())

df.describe(include='all')



# Realistic Scraped-Data Issues

The dataset intentionally contains:

- missing values
- duplicated rows
- corrupted labels
- scraping artifacts
- invalid categories
- outliers
- noisy text

These problems simulate real-world web scraping challenges.


In [ ]:

# REMOVE DUPLICATES

print("Before:", len(df))

df.drop_duplicates(inplace=True)

print("After:", len(df))


In [ ]:

# HANDLE MISSING VALUES

numeric_cols = df.select_dtypes(
    include=['int64','float64']
).columns

categorical_cols = df.select_dtypes(
    include=['object']
).columns

# NUMERICAL IMPUTATION

num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(
    df[numeric_cols]
)

# CATEGORICAL IMPUTATION

cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(
    df[categorical_cols]
)

print(df.isnull().sum().sum())


In [ ]:

# CLEAN SCRAPING ARTIFACTS

invalid_values = [
    '[deleted]',
    '[removed]',
    'NULL',
    '404_error',
    'scrape_error',
    '{}',
    '[]',
    'NoneType',
    '???',
    'missing'
]

for col in df.select_dtypes(include='object').columns:

    df[col] = df[col].astype(str)

    for val in invalid_values:

        df[col] = df[col].str.replace(
            val,
            'unknown',
            regex=False
        )

df.head()


In [ ]:

# HANDLE OUTLIERS

outlier_cols = [
    'user_age',
    'user_bmi',
    'sleep_hours',
    'target_duration_min'
]

for col in outlier_cols:

    if col in df.columns:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

print(df.shape)


In [ ]:

# CLEAN TEXT WHITESPACES

for col in df.select_dtypes(include='object').columns:

    df[col] = df[col].str.strip()

df.head()



# Multi-Class Workout Normalization

The original scraped labels contain noisy categories such as:

- CARDIOOO
- strength??
- mixed_type
- missing

These labels are normalized into clean recommendation classes.


In [ ]:

# NORMALIZE WORKOUT TYPES

def normalize_workout_type(x):

    x = str(x).lower()

    if any(k in x for k in [
        'cardio',
        'running',
        'cycling'
    ]):
        return 'cardio'

    elif any(k in x for k in [
        'strength',
        'weight',
        'lifting',
        'bodybuilding'
    ]):
        return 'strength'

    elif any(k in x for k in [
        'yoga',
        'mobility',
        'stretch'
    ]):
        return 'yoga'

    elif any(k in x for k in [
        'hiit',
        'interval'
    ]):
        return 'hiit'

    elif any(k in x for k in [
        'bodyweight',
        'calisthenics'
    ]):
        return 'bodyweight'

    elif any(k in x for k in [
        'crossfit',
        'functional'
    ]):
        return 'crossfit'

    else:
        return 'other'

df['target_workout_type'] = (
    df['target_workout_type']
    .apply(normalize_workout_type)
)

print(df['target_workout_type'].value_counts())


In [ ]:

# REMOVE VERY SMALL CLASSES

class_counts = df[
    'target_workout_type'
].value_counts()

valid_classes = class_counts[
    class_counts >= 50
].index

df = df[
    df['target_workout_type']
    .isin(valid_classes)
]

print(df['target_workout_type'].value_counts())


In [ ]:

# TARGET DISTRIBUTION

plt.figure(figsize=(8,5))

sns.countplot(
    x=df['target_workout_type']
)

plt.xticks(rotation=45)

plt.title('Workout Class Distribution')

plt.show()


In [ ]:

# ENCODE TARGET

target_encoder = LabelEncoder()

df['target_encoded'] = (
    target_encoder.fit_transform(
        df['target_workout_type']
    )
)

print(target_encoder.classes_)


In [ ]:

# FEATURE SELECTION

drop_cols = [
    'target_workout_type',
    'target_encoded'
]

FEATURES = [
    col for col in df.columns
    if col not in drop_cols
]

X = df[FEATURES]

y = df['target_encoded']

print(X.shape)
print(y.shape)


In [ ]:

# ENCODE CATEGORICAL FEATURES

categorical_cols = X.select_dtypes(
    include='object'
).columns

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

X.head()


In [ ]:

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


In [ ]:

# FEATURE SCALING

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


In [ ]:

# MACHINE LEARNING MODELS

models = {

    'Logistic Regression':
        LogisticRegression(
            max_iter=1000
        ),

    'Decision Tree':
        DecisionTreeClassifier(
            max_depth=8,
            random_state=42
        ),

    'Random Forest':
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42
        ),

    'XGBoost':
        XGBClassifier(
            objective='multi:softprob',
            num_class=6,
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            eval_metric='mlogloss',
            random_state=42
        )
}


In [ ]:

# TRAIN AND EVALUATE MODELS

results = []

for name, model in models.items():

    print('=' * 60)
    print(name)
    print('=' * 60)

    if name == 'Logistic Regression':

        model.fit(
            X_train_scaled,
            y_train
        )

        preds = model.predict(
            X_test_scaled
        )

    else:

        model.fit(
            X_train,
            y_train
        )

        preds = model.predict(
            X_test
        )

    accuracy = accuracy_score(
        y_test,
        preds
    )

    precision = precision_score(
        y_test,
        preds,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        preds,
        average='weighted'
    )

    f1 = f1_score(
        y_test,
        preds,
        average='weighted'
    )

    print(
        classification_report(
            y_test,
            preds
        )
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })


In [ ]:

# RESULTS TABLE

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1',
    ascending=False
)

results_df


In [ ]:

# BEST MODEL

best_model_name = results_df.iloc[0]['Model']

print("Best Model:", best_model_name)


In [ ]:

# FINAL XGBOOST MODEL

final_model = XGBClassifier(
    objective='multi:softprob',
    num_class=6,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=42
)

final_model.fit(
    X_train,
    y_train
)

final_preds = final_model.predict(
    X_test
)

print(
    classification_report(
        y_test,
        final_preds
    )
)


In [ ]:

# CONFUSION MATRIX

cm = confusion_matrix(
    y_test,
    final_preds
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix')

plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()


In [ ]:

# FEATURE IMPORTANCE

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': final_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

importance_df.head(20)


In [ ]:

# FEATURE IMPORTANCE PLOT

plt.figure(figsize=(10,8))

sns.barplot(
    data=importance_df.head(15),
    x='Importance',
    y='Feature'
)

plt.title(
    'Top Feature Importance'
)

plt.show()


In [ ]:

# SHAP EXPLAINABILITY

explainer = shap.TreeExplainer(
    final_model
)

shap_values = explainer.shap_values(
    X_test
)

shap.summary_plot(
    shap_values,
    X_test
)


In [ ]:

# CROSS VALIDATION

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    final_model,
    X,
    y,
    cv=cv,
    scoring='f1_weighted'
)

print(cv_scores)

print(
    'Mean Weighted F1:',
    cv_scores.mean()
)


In [ ]:

# SAVE MODEL

joblib.dump(
    final_model,
    'multiclass_workout_model.pkl'
)

joblib.dump(
    scaler,
    'scaler.pkl'
)

joblib.dump(
    encoders,
    'encoders.pkl'
)

print('Model saved successfully.')



# Expected Findings

Expected results:

- XGBoost should outperform baseline models.
- Preprocessing significantly improves dataset quality.
- BMI and fitness goals are major predictors.
- Noisy scraped data can support recommendation systems after cleaning.
- Multi-class recommendation is more realistic than binary classification.
- Weighted F1-score should realistically fall between 80%–90%.
